# HIVE-COTE v2 variance-guard failure

`hivecote-1h` fails on several UCR datasets with:

```
ValueError: Input collection has too little variation: std <= 1e-07 for N case/channel pair(s)
```

This notebook isolates the cause with stock aeon `HIVECOTEV2`:
1. the dataset is clean (no constant series),
2. plain `HIVECOTEV2` reproduces the error — it's not our benchmark code,
3. the flat array is an *internally generated* interval / shapelet subsequence that aeon 1.4's `check_collection_variance` guard rejects (inside the DrCIF / STC components),
4. tiny jitter works around it.

In [ ]:
import aeon
import numpy as np
from tscbench.utils import load_ucr_fold
from aeon.classification.hybrid import HIVECOTEV2

print("aeon", aeon.__version__)

# Datasets seen failing in the hivecote-1h run. Edit freely.
FAILING = ["Car", "UWaveGestureLibraryAll", "Plane", "ElectricDevices"]
FOLD = 3
N_JOBS = 8  # run_benchmark.py default (-j / --n-jobs)


def make_hc2():
    """EXACT hivecote-1h config from scripts/run_benchmark.py get_model().

        HIVECOTEV2(random_state=fold, n_jobs=n_jobs, time_limit_in_minutes=60)

    random_state is the fold (as in the benchmark), so this reproduces the
    same run that failed. The fit raises on the variance guard well before the
    60-minute contract matters.
    """
    return HIVECOTEV2(
        random_state=FOLD,
        n_jobs=N_JOBS,
        time_limit_in_minutes=60,
    )

## 1. The raw data is clean

Check whether any *full series* is constant (`std <= 1e-7`). Empty result = the dataset itself is fine, so the guard must trip on something HC2 generates internally.

In [ ]:
Xtr, ytr, Xte, yte = load_ucr_fold("Car", FOLD)
print("Car fold", FOLD, "train shape:", Xtr.shape)

stds = Xtr.std(axis=2)[:, 0]
flat = np.where(stds <= 1e-7)[0]
print("full-series cases with std<=1e-7:", flat.tolist())
print("min full-series std:", stds.min())

## 2. Stock HIVE-COTE v2 reproduces the failure

No benchmark code — just `HIVECOTEV2` from aeon on the clean data above. The case indices it reports as flat are *not* the constant cases from step 1 (there are none): they're internally sliced intervals / shapelet subsequences.

In [ ]:
try:
    make_hc2().fit(Xtr, ytr)
    print("HIVECOTEV2 fit: OK")
except Exception as e:
    print("HIVECOTEV2 fit FAILED ->", type(e).__name__)
    print(str(e))

## 3. Which datasets fail (clean data + stock HC2)

For each candidate: confirm no constant full series, then try a stock HC2 fit with the benchmark config. `flat_series=0` + `fit=FAILED` is the signature of this bug.

**Warning:** this is the full `hivecote-1h` config (`time_limit_in_minutes=60`). Datasets that *don't* fail will train a real HC2 and can take a long time (up to the 1-hour contract, and `ElectricDevices` is large). The failing ones raise quickly. Keep the list short, or drop large datasets when you only want to confirm the failure.

In [ ]:
for name in FAILING:
    try:
        Xtr, ytr, _, _ = load_ucr_fold(name, FOLD)
    except Exception as e:
        print(f"{name:28s} load failed: {e}")
        continue
    n_flat = int((Xtr.std(axis=2)[:, 0] <= 1e-7).sum())
    try:
        make_hc2().fit(Xtr, ytr)
        status = "OK"
    except Exception as e:
        status = f"FAILED: {type(e).__name__}"
    print(f"{name:28s} flat_series={n_flat:3d}  hc2_fit={status}")

## 4. Jitter workaround

Adding tiny noise (`1e-6`, far below the signal scale of normalized UCR series) pushes every interval / subsequence std above the `1e-7` threshold without meaningfully changing accuracy. Re-run the previously failing fit with jittered data.

In [ ]:
Xtr, ytr, Xte, yte = load_ucr_fold("Car", FOLD)

rng = np.random.default_rng(FOLD)
eps = 1e-6
Xtr_j = Xtr + rng.normal(0, eps, Xtr.shape)

try:
    make_hc2().fit(Xtr_j, ytr)
    print(f"HIVECOTEV2 fit on jittered data (eps={eps}): OK")
except Exception as e:
    print("still FAILED ->", type(e).__name__, str(e)[:120])